In [15]:
from google.maps import places_v1

In [7]:
import json
import time
import requests

# --- CONFIGURATION ---
API_KEY = "API_KEY"
SEARCH_QUERY = "restaurants"
FILE_PATH = "sd_restaurants.json"
URL = "https://places.googleapis.com/v1/places:searchText"

FIELD_MASK = ",".join([
    "places.id",
    "places.location",
    "places.displayName.text",
    "places.types",
    "places.regularOpeningHours.periods",
    "places.rating",
    "places.accessibilityOptions",
    "places.googleMapsUri",
    "places.primaryType",
    "places.priceRange",
    "nextPageToken",
])

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": FIELD_MASK,
}

# --- SAN DIEGO BOUNDING BOX GRID ---
sd_grid = {
    # --- CONVOY (4-Quadrant Split for Extreme Density) ---
    "convoy_nw": { 
        "low": {"latitude": 32.825, "longitude": -117.165},
        "high": {"latitude": 32.835, "longitude": -117.155}
    },
    "convoy_ne": { 
        "low": {"latitude": 32.825, "longitude": -117.155},
        "high": {"latitude": 32.835, "longitude": -117.145}
    },
    "convoy_sw": { 
        "low": {"latitude": 32.815, "longitude": -117.165},
        "high": {"latitude": 32.825, "longitude": -117.155}
    },
    "convoy_se": { 
        "low": {"latitude": 32.815, "longitude": -117.155},
        "high": {"latitude": 32.825, "longitude": -117.145}
    },

    # --- DOWNTOWN (Split for Vertical Density) ---
    "downtown_gaslamp_south": { 
        "low": {"latitude": 32.705, "longitude": -117.165},
        "high": {"latitude": 32.715, "longitude": -117.155}
    },
    "downtown_core_north": { 
        "low": {"latitude": 32.715, "longitude": -117.165},
        "high": {"latitude": 32.725, "longitude": -117.155}
    },
    "downtown_east_village": { 
        "low": {"latitude": 32.705, "longitude": -117.155},
        "high": {"latitude": 32.718, "longitude": -117.145}
    },

    # --- LITTLE ITALY (Split for High Concentration) ---
    "little_italy_south": { 
        "low": {"latitude": 32.719, "longitude": -117.173},
        "high": {"latitude": 32.723, "longitude": -117.165}
    },
    "little_italy_north": { 
        "low": {"latitude": 32.723, "longitude": -117.173},
        "high": {"latitude": 32.728, "longitude": -117.165}
    },

    # --- NORTH PARK (Split East/West at 30th St) ---
    "north_park_west": { 
        "low": {"latitude": 32.740, "longitude": -117.140},
        "high": {"latitude": 32.755, "longitude": -117.130}
    },
    "north_park_east": { 
        "low": {"latitude": 32.740, "longitude": -117.130},
        "high": {"latitude": 32.755, "longitude": -117.120}
    },

    # --- PACIFIC BEACH (Split East/West on Garnet) ---
    "pb_west": { 
        "low": {"latitude": 32.790, "longitude": -117.260},
        "high": {"latitude": 32.805, "longitude": -117.245}
    },
    "pb_east": { 
        "low": {"latitude": 32.795, "longitude": -117.245},
        "high": {"latitude": 32.810, "longitude": -117.225}
    },

    # --- HILLCREST (Split East/West at 5th Ave) ---
    "hillcrest_west": { 
        "low": {"latitude": 32.740, "longitude": -117.165},
        "high": {"latitude": 32.755, "longitude": -117.160}
    },
    "hillcrest_east": { 
        "low": {"latitude": 32.740, "longitude": -117.160},
        "high": {"latitude": 32.755, "longitude": -117.140}
    },

    # --- EXISTING UNTOUCHED BOXES ---
    "la_jolla_village": {
        "low": {"latitude": 32.835, "longitude": -117.285},
        "high": {"latitude": 32.855, "longitude": -117.260}
    },
    "mira_mesa": {
        "low": {"latitude": 32.905, "longitude": -117.165},
        "high": {"latitude": 32.930, "longitude": -117.125}
    },
    "ocean_beach": {
        "low": {"latitude": 32.740, "longitude": -117.255},
        "high": {"latitude": 32.755, "longitude": -117.235}
    },
    "ucsd_utc": {
        "low": {"latitude": 32.865, "longitude": -117.230},
        "high": {"latitude": 32.880, "longitude": -117.200}
    },

    # --- NEW HIGH-VALUE ADDITIONS ---
    "old_town": {
        "low": {"latitude": 32.748, "longitude": -117.198},
        "high": {"latitude": 32.758, "longitude": -117.188}
    },
    "liberty_station": {
        "low": {"latitude": 32.730, "longitude": -117.220},
        "high": {"latitude": 32.745, "longitude": -117.205}
    },
    "adams_avenue": { 
        "low": {"latitude": 32.758, "longitude": -117.130},
        "high": {"latitude": 32.768, "longitude": -117.100}
    },
    "mission_valley": { 
        "low": {"latitude": 32.760, "longitude": -117.180},
        "high": {"latitude": 32.775, "longitude": -117.140}
    },
    "carmel_valley": { 
        "low": {"latitude": 32.940, "longitude": -117.245},
        "high": {"latitude": 32.960, "longitude": -117.220}
    },
    "chula_vista_third_ave": { 
        "low": {"latitude": 32.635, "longitude": -117.085},
        "high": {"latitude": 32.650, "longitude": -117.070}
    }
}

# --- EXECUTION LOGIC ---
all_results = []
seen_place_ids = set()
max_pages = 3

print("Starting San Diego Restaurant Data Collection...\n" + "-"*50)

for neighborhood, coords in sd_grid.items():
    print(f"\nScanning neighborhood: {neighborhood.upper()}")
    
    location_restriction = {"rectangle": coords}
    next_page_token = None
    neighborhood_count = 0

    for page_idx in range(max_pages):
        payload = {
            "textQuery": SEARCH_QUERY,
            "pageSize": 20,
            "locationRestriction": location_restriction,
        }

        if next_page_token:
            payload["pageToken"] = next_page_token

        try:
            resp = requests.post(URL, headers=headers, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for {neighborhood} (Page {page_idx + 1}): {e}")
            break # Skip to the next neighborhood on failure

        places = data.get("places", [])
        
        # Deduplication loop
        new_places_in_page = 0
        for place in places:
            place_id = place.get("id")
            if place_id not in seen_place_ids:
                seen_place_ids.add(place_id)
                all_results.append(place)
                new_places_in_page += 1
                neighborhood_count += 1

        print(f"   Page {page_idx + 1}: Found {len(places)} places ({new_places_in_page} new)")

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(2)

    if neighborhood_count >= 60:
        print(f"   WARNING: {neighborhood.upper()} maxed out at {neighborhood_count} places. Consider making this bounding box smaller!")

# --- SAVE TO FILE ---
print("\n" + "-"*50)
print(f"Scraping complete! Writing {len(all_results)} unique restaurants to {FILE_PATH}...")

with open(FILE_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("Done!")

Starting San Diego Restaurant Data Collection...
--------------------------------------------------

📍 Scanning neighborhood: CONVOY_NW
   Page 1: Found 18 places (18 new)

📍 Scanning neighborhood: CONVOY_NE
   Page 1: Found 20 places (20 new)
   Page 2: Found 19 places (19 new)

📍 Scanning neighborhood: CONVOY_SW
   Page 1: Found 20 places (20 new)
   Page 2: Found 15 places (15 new)

📍 Scanning neighborhood: CONVOY_SE
   Page 1: Found 20 places (20 new)
   Page 2: Found 9 places (9 new)

📍 Scanning neighborhood: DOWNTOWN_GASLAMP_SOUTH
   Page 1: Found 20 places (20 new)
   Page 2: Found 20 places (20 new)
   Page 3: Found 20 places (20 new)
   ⚠️ WARNING: DOWNTOWN_GASLAMP_SOUTH maxed out at 60 places. Consider making this bounding box smaller!

📍 Scanning neighborhood: DOWNTOWN_CORE_NORTH
   Page 1: Found 20 places (20 new)
   Page 2: Found 20 places (20 new)
   Page 3: Found 11 places (11 new)

📍 Scanning neighborhood: DOWNTOWN_EAST_VILLAGE
   Page 1: Found 20 places (20 new)
   Pag